In [0]:
from pyspark.sql import functions as F

CATALOG = "civic_signal_dbx_dev"

opp = spark.table(f"{CATALOG}.silver.opportunities")
buyers = spark.table(f"{CATALOG}.silver.buyers")
categories = spark.table(f"{CATALOG}.silver.categories")
quarantine = spark.table(f"{CATALOG}.silver.opportunities_quarantine")

opp_count = opp.count()
buyer_count = buyers.count()
category_count = categories.count()
quarantine_count = quarantine.count()

opp_duplicates = (
    opp.groupBy("opportunity_id")
       .count()
       .filter("count > 1")
       .count()
)

buyer_duplicates = (
    buyers.groupBy("buyer_id")
          .count()
          .filter("count > 1")
          .count()
)

null_opp_keys = (
    opp.filter(
        F.col("opportunity_id").isNull() |
        (F.trim("opportunity_id") == "")
    ).count()
)

null_buyer_keys = (
    buyers.filter(
        F.col("buyer_id").isNull() |
        (F.trim("buyer_id") == "")
    ).count()
)

assert opp_count > 0, "Silver opportunities must not be empty"
assert buyer_count > 0, "Silver buyers must not be empty"
assert category_count > 0, "Silver categories must not be empty"

assert opp_duplicates == 0, \
    f"Duplicate opportunity_id values found: {opp_duplicates}"

assert buyer_duplicates == 0, \
    f"Duplicate buyer_id values found: {buyer_duplicates}"

assert null_opp_keys == 0, \
    f"Null/blank opportunity_id values found: {null_opp_keys}"

assert null_buyer_keys == 0, \
    f"Null/blank buyer_id values found: {null_buyer_keys}"

print("Silver validation PASSED")
print(f"opportunities: {opp_count}")
print(f"buyers: {buyer_count}")
print(f"categories: {category_count}")
print(f"quarantine: {quarantine_count}")
print(f"opportunity duplicates: {opp_duplicates}")
print(f"buyer duplicates: {buyer_duplicates}")

Silver validation PASSED
opportunities: 12
buyers: 6
categories: 7
quarantine: 3
opportunity duplicates: 0
buyer duplicates: 0


In [0]:
print(
    "Bronze opportunities:",
    spark.table(f"{CATALOG}.bronze.opportunities_raw").count()
)

print(
    "Silver opportunities:",
    spark.table(f"{CATALOG}.silver.opportunities").count()
)

print(
    "Silver quarantine:",
    spark.table(f"{CATALOG}.silver.opportunities_quarantine").count()
)

print(
    "Silver buyers:",
    spark.table(f"{CATALOG}.silver.buyers").count()
)

print(
    "Silver categories:",
    spark.table(f"{CATALOG}.silver.categories").count()
)

print(
    "funding_source in Bronze:",
    "funding_source" in spark.table(
        f"{CATALOG}.bronze.opportunities_raw"
    ).columns
)

print(
    "funding_source in Silver:",
    "funding_source" in spark.table(
        f"{CATALOG}.silver.opportunities"
    ).columns
)

Bronze opportunities: 18
Silver opportunities: 12
Silver quarantine: 3
Silver buyers: 6
Silver categories: 7
funding_source in Bronze: True
funding_source in Silver: False
